### **Student Information**
GitHub ID: vejasovaite

Kaggle name: Veja Sovaite

Kaggle private scoreboard snapshot: (I accidentally missed the Kaggle competition submission deadline. Therefore, I don't have a private leaderboard score. My final model achieved a macro F1 of 0.6140 on the local validation set.)

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

**Syntax:** `#` creates the largest heading (H1).

---
**Syntax:** `---` creates a horizontal rule (a separator line).

## 1. Model Development (10 pts Required)

**Syntax:** `##` creates a secondary heading (H2).

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

First, I loaded the JSON and CSV files and merged them based on the post ID. The dataset consists of 3360 labeled training samples and 1440 test samples across six emotion classes: anger, disgust, fear, joy, sadness and surprise. Before training the models, I performed the following preprocessing steps:

- Converted the text to lowercase.
- Replaced URLs and user mentions with common placeholders.
- Removed unnecessary whitespaces.
- Kept punctuation, hashtags, emojis and stop words, as I thought they could contain relevant information for emotion classification.

I also noticed that the dataset was imbalanced, with joy being the most frequent class (875 samples) and fear being the least frequent class (288 samples). When checking for duplicates, I found 50 repeated training text entries, 16 unique texts with conflicting labels and 59 unique texts that appeared in both the training and test sets. Therefore, instead of using a normal random split, I decided to use `StratifiedGroupKFold` to create an 80/20 training-validation split. This allowed me to keep identical texts in the same subset while approximately preserving the class distribution. The resulting split consisted of 2688 training and 672 validation samples, with no identical cleaned texts shared between them. I used the same validation split throughout all experiments to make the results comparable.

### 1.2 Feature Engineering Steps

**TF-IDF**

Since Linear SVM with TF-IDF performed best in the Master notebook, I decided to start with a similar approach. However, as this dataset contains six emotions and texts from different sources, I was not sure whether the same representation would perform equally well. I compared three different feature representations:

- Word-level TF-IDF using unigrams and bigrams.
- Character-level TF-IDF using character n-grams of length 3 to 5.
- A combination of word-level and character-level TF-IDF.

The combined representation performed slightly better than either representation alone. I also tested different regularization strengths and class weights, which improved the macro F1 from 0.4842 to 0.5013.

**Semantic embeddings**

Since TF-IDF mainly captures lexical information, I wanted to see whether including semantic information could improve the results. For this, I used the pretrained `all-MiniLM-L6-v2` model to represent each text as a 384-dimensional embedding. I then trained a Linear SVM on these embeddings and achieved a macro F1 of 0.5188, which was better than using TF-IDF alone.

**Combining lexical and semantic features**

Since both representations capture different types of information, I decided to combine the word-level TF-IDF, character-level TF-IDF and MiniLM embeddings. I experimented with different relative weights for the embeddings and different regularization strengths. The best result was achieved with an embedding weight of 1.0 and C = 0.1, resulting in a macro F1 of 0.5622.

### 1.3 Explanation of Your Model

**Transformer models**

Although combining lexical and semantic features improved the results, I wanted to see whether fine-tuning a transformer directly on the emotion classification task would perform better. I first fine-tuned RoBERTa-base for four epochs. Unlike in the previous experiments, I used the original text, since capitalization and punctuation might also contain relevant emotional information. The best result was achieved after three epochs, with a macro F1 of 0.5941. Based on this improvement, I decided to try DeBERTa-v3-base as well. I fine-tuned the model for five epochs, using a learning rate of 2e-5 and selecting the best checkpoint based on validation macro F1. DeBERTa achieved a macro F1 of 0.6140 and an accuracy of 0.6265.

**Additional experiments**

Since DeBERTa's macro F1 was still increasing after five epochs, I wanted to check whether longer training would further improve the results. I therefore tested a longer training schedule of up to ten epochs with early stopping. However, the best macro F1 was 0.6131, and training stopped after six epochs. The validation loss also continued to increase, suggesting that the model was starting to overfit.

I also tested different combinations of RoBERTa and DeBERTa prediction probabilities. Although the two models made different errors on 99 validation samples, combining their predictions did not outperform DeBERTa alone.

**Error analysis and limitations**

Based on the validation results, joy was the easiest emotion to classify, with an F1 score of 0.77, while disgust was the most difficult, with an F1 score of 0.47. This might be because some emotions, especially anger and disgust, share similar negative expressions and are therefore more difficult to distinguish. Compared to the hybrid model, the final DeBERTa model also predicted minority classes more frequently, especially disgust and fear. However, without the test labels, it is not possible to determine whether this actually improved test performance. Overall, I was able to improve the macro F1 from 0.4614 with the initial TF-IDF model to 0.6140 with DeBERTa. However, since I used the same validation set for multiple experiments and model selection, the reported performance might be slightly optimistic. Additionally, since I missed the Kaggle competition deadline, I was not able to evaluate my final predictions on the private leaderboard. All reported scores are therefore based on my local validation set.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

Throughout this homework, I used ChatGPT to discuss possible approaches.

Some of my main prompts were:

- Combining different features: AI suggested combining TF-IDF with semantic embeddings. I decided to test this since both representations capture different information, which improved the macro F1 to 0.5622.
- AI suggested exploring stronger transformer models and ensembling. Fine-tuning DeBERTa improved my results, while ensembling did not.
- AI suggested longer training with early stopping. However, this did not improve the results, so I kept the original five-epoch configuration.
  
Overall, I found AI useful for exploring different approaches, but not all suggestions improved the results. Therefore, I selected the final model based on the experimental results.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
### Add the code related to the preprocessing steps in cells inside this section
import json
from pathlib import Path

SEED = 42
DATA_DIR = Path("/kaggle/input/competitions/2026-dm-lab-2-private-competition")

# loaded datasets
with open(DATA_DIR / "final_posts.json", encoding="utf-8") as f:
    raw_posts = json.load(f)

emotions = pd.read_csv(DATA_DIR / "emotion.csv")
identification = pd.read_csv(DATA_DIR / "data_identification.csv")
sample_submission = pd.read_csv(DATA_DIR / "sampleSubmission.csv")

# inspected JSON structure
print("JSON type:", type(raw_posts))
print("Number of entries:", len(raw_posts))

if isinstance(raw_posts, list):
    print(raw_posts[0])
elif isinstance(raw_posts, dict):
    print(list(raw_posts.items())[:1])

# inspected CSV files
for name, df in [
    ("Emotions", emotions),
    ("Identification", identification),
    ("Sample submission", sample_submission)
]:
    print(f"\n{name}: {df.shape}")
    display(df.head())

print("\nClass distribution:")
print(emotions.iloc[:, -1].value_counts())

print("\nTrain/test distribution:")
print(identification["set"].value_counts())

import re

# merged the three datasets
posts = pd.DataFrame(raw_posts)
df = posts.merge(identification, on="post_id", validate="one_to_one")
df = df.merge(emotions, on="post_id", how="left", validate="one_to_one")
train = df[df["set"] == "train"].copy()
test = df[df["set"] == "test"].copy()

# text cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
train["clean"] = train["text"].fillna("").apply(clean_text)
test["clean"] = test["text"].fillna("").apply(clean_text)
print("Training shape:", train.shape)
print("Test shape:", test.shape)
print("Missing training labels:", train["label"].isna().sum())
print("Repeated training texts:", train["clean"].duplicated().sum())
display(train.head())

# checked repeated texts and conflicting labels
conflicts = train.groupby("clean")["label"].nunique()
print("Texts with conflicting labels:", (conflicts > 1).sum())

# checked overlap between training and test sets
shared = set(train["clean"]) & set(test["clean"])
print("Shared texts between train and test:", len(shared))

## 2. Feature Engineering Steps

In [ ]:
### Add the code related to the feature engineering steps in cells inside this section
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)
train_idx, val_idx = next(
    splitter.split(train["clean"], train["label"], groups=train["clean"])
)
X_train = train.iloc[train_idx]["clean"]
X_val = train.iloc[val_idx]["clean"]
y_train = train.iloc[train_idx]["label"]
y_val = train.iloc[val_idx]["label"]
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Shared texts:", len(set(X_train) & set(X_val)))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report

# word-level features
def word_features():
    return TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )

# character-level features
def char_features():
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True
    )

# compared three feature representations
representations = {
    "Word TF-IDF": word_features(),
    "Character TF-IDF": char_features(),
    "Combined TF-IDF": FeatureUnion([
        ("word", word_features()),
        ("char", char_features())
    ])
}
results = []
models = {}

# trained and evaluated each model
for name, features in representations.items():
    model = Pipeline([
        ("features", features),
        ("classifier", LinearSVC(random_state=SEED))
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    results.append({
        "Representation": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro F1": f1_score(y_val, predictions, average="macro")
    })
    models[name] = model

# compared results
results_df = pd.DataFrame(results).sort_values(
    "Macro F1", ascending=False
)
display(results_df.round(4))
# evaluated the best-performing model
best_name = results_df.iloc[0]["Representation"]
best_model = models[best_name]
print("Best model:", best_name)
print(classification_report(y_val, best_model.predict(X_val)))

tuning_results = []
tuned_models = {}

for C in [0.1, 0.5, 1.0, 2.0, 5.0]:
    for weights in [None, "balanced"]:
        model = Pipeline([
            ("features", FeatureUnion([
                ("word", word_features()),
                ("char", char_features())
            ])),
            ("classifier", LinearSVC(
                C=C,
                class_weight=weights,
                random_state=SEED
            ))
        ])
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)
        tuning_results.append({
            "C": C,
            "Class weights": str(weights),
            "Accuracy": accuracy_score(y_val, predictions),
            "Macro F1": f1_score(
                y_val, predictions, average="macro"
            )
        })
        tuned_models[(C, str(weights))] = model
tuning_df = pd.DataFrame(tuning_results).sort_values(
    "Macro F1", ascending=False
)
display(tuning_df.round(4))
best_row = tuning_df.iloc[0]
best_model = tuned_models[
    (best_row["C"], best_row["Class weights"])
]
print(classification_report(
    y_val, best_model.predict(X_val)
))

from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = encoder.encode(
    X_train.tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)
X_val_emb = encoder.encode(
    X_val.tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)
print(X_train_emb.shape)
print(X_val_emb.shape)

embedding_results = []
embedding_models = {}

for C in [0.1, 1.0, 5.0, 10.0]:
    for weights in [None, "balanced"]:
        model = LinearSVC(
            C=C,
            class_weight=weights,
            random_state=SEED
        )
        model.fit(X_train_emb, y_train)
        predictions = model.predict(X_val_emb)
        embedding_results.append({
            "C": C,
            "Class weights": str(weights),
            "Accuracy": accuracy_score(y_val, predictions),
            "Macro F1": f1_score(
                y_val, predictions, average="macro"
            )
        })
        embedding_models[(C, str(weights))] = model
embedding_df = pd.DataFrame(embedding_results).sort_values(
    "Macro F1", ascending=False
)
display(embedding_df.round(4))

from scipy.sparse import hstack, csr_matrix

# extracted lexical features
tfidf = FeatureUnion([
    ("word", word_features()),
    ("char", char_features())
])

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

hybrid_results = []
hybrid_models = {}

# compared different relative weights of semantic features
for weight in [0.5, 1.0, 2.0, 4.0]:

    X_train_hybrid = hstack([
        X_train_tfidf,
        csr_matrix(X_train_emb * weight)
    ]).tocsr()

    X_val_hybrid = hstack([
        X_val_tfidf,
        csr_matrix(X_val_emb * weight)
    ]).tocsr()

    for C in [0.01, 0.1, 0.5]:
        model = LinearSVC(C=C, random_state=SEED)
        model.fit(X_train_hybrid, y_train)

        predictions = model.predict(X_val_hybrid)

        hybrid_results.append({
            "Embedding weight": weight,
            "C": C,
            "Accuracy": accuracy_score(y_val, predictions),
            "Macro F1": f1_score(
                y_val, predictions, average="macro"
            )
        })

        hybrid_models[(weight, C)] = model

hybrid_df = pd.DataFrame(hybrid_results).sort_values(
    "Macro F1", ascending=False
)

display(hybrid_df.round(4))

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

best_hybrid_model = hybrid_models[(1.0, 0.1)]

X_val_best = hstack([
    X_val_tfidf,
    csr_matrix(X_val_emb)
]).tocsr()

val_predictions = best_hybrid_model.predict(X_val_best)

print(classification_report(y_val, val_predictions))

ConfusionMatrixDisplay.from_predictions(
    y_val,
    val_predictions,
    cmap="Blues",
    xticks_rotation=45
)

plt.title("Hybrid model confusion matrix")
plt.tight_layout()
plt.show()

## 3. Model Implementation Steps

In [ ]:
### Add the code related to the model implementation steps in cells inside this section
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(train["label"])
MODEL_NAME = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# used original text because capitalization and punctuation
# can provide additional emotional information
def prepare_dataset(indices):
    subset = train.iloc[indices]

    dataset = Dataset.from_dict({
        "text": subset["text"].fillna("").tolist(),
        "labels": label_encoder.transform(
            subset["label"]
        ).tolist()
    })

    return dataset.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            max_length=128
        ),
        batched=True,
        remove_columns=["text"]
    )

train_dataset = prepare_dataset(train_idx)
val_dataset = prepare_dataset(val_idx)
print(train_dataset)
print(val_dataset)
print(label_encoder.classes_)

from transformers import set_seed

set_seed(SEED)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels, predictions, average="macro"
        )
    }
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6
)
training_args = TrainingArguments(
    output_dir="/kaggle/working/roberta_emotions",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=68,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    report_to="none",
    seed=SEED
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)
trainer.train()

results = trainer.evaluate()
print(results)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
roberta_output = trainer.predict(val_dataset)
roberta_logits = roberta_output.predictions
roberta_predictions = label_encoder.inverse_transform(
    np.argmax(roberta_logits, axis=-1)
)
print(classification_report(y_val, roberta_predictions))

roberta_trainer = trainer
roberta_tokenizer = tokenizer
roberta_f1 = f1_score(
    y_val, roberta_predictions, average="macro"
)
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = prepare_dataset(train_idx)
val_dataset = prepare_dataset(val_idx)

import gc
from transformers import EarlyStoppingCallback

set_seed(SEED)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6
).float()

print("Model dtype:", next(model.parameters()).dtype)

training_args = TrainingArguments(
    output_dir="/kaggle/working/deberta",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=20,
    fp16=True,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

deberta_trainer = trainer
deberta_tokenizer = tokenizer
deberta_output = trainer.predict(val_dataset)
deberta_logits = deberta_output.predictions
deberta_predictions = label_encoder.inverse_transform(
    np.argmax(deberta_logits, axis=-1)
)
print(classification_report(y_val, deberta_predictions))

from scipy.special import softmax

roberta_probs = softmax(roberta_logits, axis=1)
deberta_probs = softmax(deberta_logits, axis=1)

ensemble_results = []

for weight in np.linspace(0, 1, 11):

    ensemble_probs = (
        weight * deberta_probs
        + (1 - weight) * roberta_probs
    )

    predictions = label_encoder.inverse_transform(
        np.argmax(ensemble_probs, axis=1)
    )

    ensemble_results.append({
        "DeBERTa weight": weight,
        "RoBERTa weight": 1 - weight,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro F1": f1_score(
            y_val, predictions, average="macro"
        )
    })

ensemble_df = pd.DataFrame(ensemble_results).sort_values(
    "Macro F1", ascending=False
)

display(ensemble_df.round(4))

roberta_correct = roberta_predictions == y_val.to_numpy()
deberta_correct = deberta_predictions == y_val.to_numpy()

print("Both correct:", np.sum(roberta_correct & deberta_correct))
print("Only RoBERTa correct:", np.sum(roberta_correct & ~deberta_correct))
print("Only DeBERTa correct:", np.sum(deberta_correct & ~roberta_correct))
print("Both incorrect:", np.sum(~roberta_correct & ~deberta_correct))